In [1]:
from cifar import load_data
from base.config import DataConfig
from resnet.resnet import ResNet18

In [2]:
dc = DataConfig(root="/Volumes/satechi/ml_projects/cifar/")

In [3]:
ds = load_data.load_dataset(dc)

/Users/peterwen/projects/learn_cv/cifar/load_data.py:38: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dict_data = pk.load(f, encoding="latin1")


In [ ]:

import torch
import torch.nn as nn


class Block(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        padding: int = 1,
        stride: int = 1,
        downsample: nn.Module | None = None,
    ):

        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
        self.shortcut = nn.Sequential()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out



class ResNet18(nn.Module):
    def __init__(self, in_channels: int, n_classes: int = 100):
        super().__init__()
        self.in_channels = in_channels
        self.stem = Stem(self.in_channels)

        self.layer1 = self._make_layer(Block, out_channels=64, blocks=2, stride=1)
        self.layer2 = self._make_layer(Block, out_channels=128, blocks=2, stride=2)
        self.layer3 = self._make_layer(Block, out_channels=256, blocks=2, stride=2)
        self.layer4 = self._make_layer(Block, out_channels=512, blocks=2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None

        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.in_channels,
                    out_channels * block.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion

        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)


In [36]:
class Stem(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(
            3, in_channels, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        return out

stem = Stem(3)

In [37]:
out = stem.conv1(ds[0][0].unsqueeze(0))
out.shape

torch.Size([1, 3, 32, 32])

In [35]:
stem.maxpool(out).shape

torch.Size([1, 3, 16, 16])